# MQR: bounded Kaggle T4 smoke and real-data calibration

Attach the private `mqr-kaggle-readiness.zip` upload as a Kaggle dataset and enable a T4 accelerator. This notebook autodetects one mounted ZIP or extracted source tree. The package includes verified prepared data and provenance, so a separate data upload is unnecessary.

Run All first completes a tiny four-step GPU smoke test for dense and MQR heads, including checkpoint reload/resume. If it passes, the default runs a bounded 120-step paired calibration on the included real data (about 2M predicted tokens per model). Set `RUN_PREPARED = False` for smoke only. The duplicate synthetic full-shape stage remains disabled. No 3B-token pretraining, corpus download or publication is performed. Read `docs/KAGGLE.md` for interpretation.


In [ ]:
from pathlib import Path
import importlib.metadata
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile

# Normally autodetected. Set a mounted directory or ZIP path only if ambiguous.
UPLOAD_ROOT = None
CODE_ROOT = Path("/kaggle/working/mqr-source")
assert sys.version_info >= (3, 11), "Python 3.11+ is required"
if UPLOAD_ROOT is None:
    mounted = Path("/kaggle/input")
    candidates = list(mounted.rglob("mqr-kaggle-readiness.zip"))
    candidates += [marker.parent for marker in mounted.rglob("pyproject.toml")
                   if (marker.parent / "src" / "mqr").is_dir()]
    if len(candidates) != 1:
        raise ValueError("Set UPLOAD_ROOT to the intended source directory or ZIP; found: "
                         + repr([str(path) for path in candidates]))
    UPLOAD_ROOT = candidates[0]
else:
    UPLOAD_ROOT = Path(UPLOAD_ROOT)
assert UPLOAD_ROOT.exists(), f"Source upload does not exist: {UPLOAD_ROOT}"
print("Selected upload:", UPLOAD_ROOT)
if not CODE_ROOT.exists():
    archives = [UPLOAD_ROOT] if UPLOAD_ROOT.is_file() else list(UPLOAD_ROOT.glob("*.zip"))
    if archives:
        assert len(archives) == 1, "Choose the exact source ZIP using UPLOAD_ROOT"
        CODE_ROOT.mkdir(parents=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for item in archive.infolist():
                target = (CODE_ROOT / item.filename).resolve()
                assert target.is_relative_to(CODE_ROOT.resolve()), "ZIP path escapes source directory"
                assert (item.external_attr >> 16) & 0o170000 != 0o120000, "ZIP symlinks are not supported"
            archive.extractall(CODE_ROOT)
    else:
        shutil.copytree(UPLOAD_ROOT, CODE_ROOT)
else:
    print("Reusing copied source; choose a fresh CODE_ROOT when changing the uploaded ZIP")
projects = list(CODE_ROOT.rglob("pyproject.toml"))
assert len(projects) == 1, "Source upload must contain one project pyproject.toml"
PROJECT = projects[0].parent
package_manifest = PROJECT / "package_manifest.json"
if package_manifest.exists():
    payload = json.loads(package_manifest.read_text(encoding="utf-8"))
    for name, expected in payload["files"].items():
        artifact = (PROJECT / name).resolve()
        assert artifact.is_relative_to(PROJECT.resolve()), "Manifest path escapes project"
        digest = hashlib.sha256()
        with artifact.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1048576), b""):
                digest.update(chunk)
        assert artifact.stat().st_size == expected["bytes"], f"Package size mismatch: {name}"
        assert digest.hexdigest() == expected["sha256"], f"Package hash mismatch: {name}"
    print("Verified packaged payload files:", len(payload["files"]))
# Use the existing CUDA torch; do not resolve or reinstall project dependencies.
assert int(importlib.metadata.version("setuptools").split(".")[0]) >= 77, "Install setuptools>=77 separately, then rerun setup"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(PROJECT)])
# Editable .pth files are processed at startup, so expose source to this live kernel.
sys.path.insert(0, str(PROJECT / "src"))
print("Installed source:", PROJECT)
print("Included prepared data:", PROJECT / "prepared")


def require_tokenizers():
    """Bootstrap only the tokenizer wheel before any tokenizers import."""
    def supported(value):
        try:
            version = tuple(map(int, value.split(".")[:2]))
        except (AttributeError, TypeError, ValueError):
            return False
        return len(version) == 2 and (0, 21) <= version < (0, 23)

    try:
        installed = importlib.metadata.version("tokenizers")
    except importlib.metadata.PackageNotFoundError:
        installed = None
    loaded = sys.modules.get("tokenizers")
    if loaded is not None and getattr(loaded, "__version__", None) != installed:
        raise RuntimeError("A different tokenizers version is already loaded; restart the clean kernel before setup")
    if not supported(installed):
        if any(name == "tokenizers" or name.startswith("tokenizers.") for name in sys.modules):
            raise RuntimeError("tokenizers is already imported; restart the clean kernel before changing its wheel")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps",
                               "--only-binary=:all:", "tokenizers==0.22.2"])
        importlib.invalidate_caches()
        installed = importlib.metadata.version("tokenizers")
        if installed != "0.22.2":
            raise RuntimeError("Tokenizer installation did not expose the requested tokenizers==0.22.2 wheel")
    if not supported(installed):
        raise RuntimeError("Prepared BPE data requires tokenizers>=0.21,<0.23")
    return installed

# Run before importing MQR or any library that could load tokenizers.
print("Tokenizers runtime:", require_tokenizers())


In [ ]:
import json
import time
import numpy as np
import torch
from mqr.kaggle import inspect_device, run_session

DEVICE = "cuda:0"  # One explicit GPU; a second T4 does not pool VRAM.
assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26), "NumPy 1.26+ is required"
assert (2, 5) <= tuple(map(int, str(torch.__version__).split(".")[:2])) < (3, 0), "PyTorch >=2.5,<3 is required"
print(json.dumps(inspect_device(DEVICE), indent=2))
RUN_ROOT = Path("/kaggle/working") / ("mqr-" + time.strftime("%Y%m%d-%H%M%S"))
print("Outputs:", RUN_ROOT)


## Default: tiny GPU correctness smoke

Both heads train for two steps, save/load on CPU to verify portability and resume state, then continue to step four on the selected GPU. The helper forces CUDA FP16 with GradScaler, which is appropriate for T4. These random-token losses are implementation checks only.


In [ ]:
smoke = run_session(stage="smoke", device=DEVICE, output_root=RUN_ROOT / "smoke")
print(json.dumps(smoke["results"], indent=2))


## Optional: duplicate synthetic full-shape calibration

This stage stays disabled because the default real-data stage below already measures the full planned shape. Enable only for a separate synthetic diagnostic. Shape: 12 layers, d512, 64k vocabulary, context 512, microbatch one, accumulation 32. A 120-step stop preserves the complete 183106-step LR schedule; steps 1–20 are discarded from steady-state timing. Synthetic tokens do not measure LM quality.


In [ ]:
RUN_CALIBRATION = False
if RUN_CALIBRATION:
    calibration = run_session(stage="calibrate", device=DEVICE,
                              output_root=RUN_ROOT / "synthetic-calibration",
                              steps=120, warmup_steps=20)
    print(json.dumps({key: value["calibration"] for key, value in calibration["results"].items()}, indent=2))


## Default: verified included real-data calibration

The four-step smoke must pass for both heads before this stage runs. `DATA_ROOT` defaults to the included `PROJECT/prepared` directory. The bundle is required by default; override only to use another attached prepared bundle. Setup preserves a compatible tokenizer runtime; if missing or outside the supported range, it installs only the tested `tokenizers==0.22.2` binary wheel without dependencies. CUDA PyTorch is unchanged.

All advertised artifact hashes are verified; actual tokenizer size is applied to both heads. Only train/validation streams are consumed. This is a bounded 120-step calibration (about 2M predicted tokens per model), not the 3B-token study or evidence of convergence. Set `RUN_PREPARED = False` for smoke only. Source balancing and leakage audits remain a separate requirement.


In [ ]:
RUN_PREPARED = True
DATA_ROOT = PROJECT / "prepared"  # Override only when using a different attached bundle.
if RUN_PREPARED:
    assert set(smoke["results"]) == {"dense", "mqr"}, "Complete the paired smoke first"
    assert all(result["training"]["completed_steps"] == 4
               and result["resume_check"]["continued_from_step_2_to"] == 4
               and result["resume_check"]["finite_weights"]
               for result in smoke["results"].values()), "Both four-step smoke/resume checks must pass"
    assert (DATA_ROOT / "manifest.json").is_file(), "Attach the readiness ZIP with its prepared data, or set RUN_PREPARED=False"
    require_tokenizers()
    prepared = run_session(stage="prepared", device=DEVICE, data_dir=DATA_ROOT,
                           output_root=RUN_ROOT / "prepared-calibration",
                           steps=120, warmup_steps=20)
    print(json.dumps({key: value["calibration"] for key, value in prepared["results"].items()}, indent=2))


## Preserve outputs

Run the archive cell, then download its ZIP from Kaggle Files before ending the session. Keep the complete source/data versions with the report. No upload or publication is performed by this notebook. Output checkpoints contain weights, optimizer, scaler, RNG, sampler and schedule state. Peak allocated VRAM in the report differs from the GPU's total reserved memory.


In [ ]:
archive_path = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
print("Download this file from Kaggle Files:", archive_path)
